# 03 — GPU Roofline and Batching (A-Day 5)

1. Arithmetic Intensity = 1 proof for decode (batch=1).
2. Roofline: batch size sweep and throughput plot.
3. Static vs continuous batching.
4. PagedAttention waste (max_num_seqs=1 vs 32).
5. KV cache quantization (FP8 vs BF16).

## 1. Arithmetic Intensity = 1 proof

One linear layer: $y = Wx$, $W \in \mathbb{R}^{d\times d}$, $x \in \mathbb{R}^d$. FLOPs = $2d^2$, Bytes = $2d^2$, AI = FLOPs/Bytes = 1. Compare to MI250X ridge point.

In [ ]:
d = 2048  # Qwen2.5-1.5B hidden size
bytes_per_param = 2  # BF16
flops = 2 * d * d
bytes_loaded = d * d * bytes_per_param
ai = flops / bytes_loaded
ridge_point_mi250x = 204  # FLOP/byte for BF16 (approx)
utilization_pct = 100 * ai / ridge_point_mi250x
print(f"Hidden dim d = {d}")
print(f"FLOPs (Wx): 2*d² = {flops}")
print(f"Bytes (load W): d² * 2 = {bytes_loaded}")
print(f"Arithmetic Intensity = FLOPs/Bytes = {ai}")
print(f"MI250X ridge point ~{ridge_point_mi250x} FLOP/byte")
print(f"Utilization (batch=1): {utilization_pct:.2f}%")

## 2. Roofline: batch size sweep

Send batch sizes [1, 2, 4, 8, 16, 32, 64]; measure tokens/sec. Plot throughput vs batch size. Compare to NB0 baseline.

In [ ]:
from vllm import LLM, SamplingParams
import time

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
llm = LLM(model=MODEL_ID, trust_remote_code=True, max_model_len=2048, max_num_seqs=64)
sampling = SamplingParams(temperature=0, max_tokens=50)

In [ ]:
batch_sizes = [1, 2, 4, 8, 16, 32, 64]
prompt_base = "Write one sentence about AI."
throughputs = []
for b in batch_sizes:
    prompts = [prompt_base] * b
    t0 = time.perf_counter()
    outs = llm.generate(prompts, sampling_params=sampling)
    elapsed = time.perf_counter() - t0
    total_tokens = sum(len(o.outputs[0].token_ids) for o in outs)
    tps = total_tokens / elapsed
    throughputs.append(tps)
    print(f"Batch {b}: {tps:.1f} tok/s")

print("Compare to your NB0 baseline (no KV cache, batch=1) to see vLLM gain.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(8, 4))
plt.plot(batch_sizes, throughputs, 'o-')
plt.xlabel("Batch size")
plt.ylabel("Throughput (tokens/s)")
plt.title("Roofline: throughput vs batch size (memory-bound -> compute-bound)")
plt.grid(True)
plt.show()

## 3. Static vs continuous batching

Simulate 8 requests of varying lengths. With vLLM we submit all and let continuous batching drain the queue; measure total wall time.

In [ ]:
prompts_varied = [
    "Say A.",
    "Say B.",
    "Explain in one word: gravity.",
    "What is 1+1?",
    "Hello.",
    "List one planet.",
    "Yes or no: is the sky blue?",
    "One sentence about ML.",
]
sampling_short = SamplingParams(temperature=0, max_tokens=5)
t0 = time.perf_counter()
outs = llm.generate(prompts_varied, sampling_params=sampling_short)
elapsed = time.perf_counter() - t0
print(f"8 requests (continuous batching): {elapsed:.2f} s total")
print(f"Total output tokens: {sum(len(o.outputs[0].token_ids) for o in outs)}")

## 4. PagedAttention: max_num_seqs impact

Compare cache utilization with max_num_seqs=1 vs 32 (more concurrent sequences -> more fragmentation without PagedAttention; with it we use blocks efficiently).

In [ ]:
# With max_num_seqs=1 we effectively run one sequence at a time (contiguous-style).
# With max_num_seqs=32 we allow many concurrent; PagedAttention avoids fragmentation.
llm_m1 = LLM(model=MODEL_ID, trust_remote_code=True, max_model_len=2048, max_num_seqs=1)
llm_m32 = LLM(model=MODEL_ID, trust_remote_code=True, max_model_len=2048, max_num_seqs=32)
prompts = ["Hello."] * 8
sp = SamplingParams(temperature=0, max_tokens=20)
t1 = time.perf_counter()
llm_m1.generate(prompts, sampling_params=sp)
t1 = time.perf_counter() - t1
t32 = time.perf_counter()
llm_m32.generate(prompts, sampling_params=sp)
t32 = time.perf_counter() - t32
print(f"max_num_seqs=1:  {t1:.2f} s")
print(f"max_num_seqs=32: {t32:.2f} s")
print("Higher max_num_seqs allows batching; PagedAttention keeps cache usage efficient.")

## 5. KV cache quantization: FP8 vs BF16

Compare `--kv-cache-dtype fp8_e5m2` vs default (bf16): throughput, TPOT, peak memory.

In [ ]:
# Default: KV cache in bf16
llm_bf16 = LLM(model=MODEL_ID, trust_remote_code=True, max_model_len=2048)
prompts = ["Explain AI in one sentence."] * 4
sp = SamplingParams(temperature=0, max_tokens=30)
t0 = time.perf_counter()
out_bf16 = llm_bf16.generate(prompts, sampling_params=sp)
time_bf16 = time.perf_counter() - t0
n_tok_bf16 = sum(len(o.outputs[0].token_ids) for o in out_bf16)
print(f"BF16 KV cache: {time_bf16:.2f} s, {n_tok_bf16} tokens, {n_tok_bf16/time_bf16:.1f} tok/s")

In [ ]:
# FP8 KV cache (if supported on your GPU)
try:
    llm_fp8 = LLM(model=MODEL_ID, trust_remote_code=True, max_model_len=2048, kv_cache_dtype="fp8_e5m2")
    t0 = time.perf_counter()
    out_fp8 = llm_fp8.generate(prompts, sampling_params=sp)
    time_fp8 = time.perf_counter() - t0
    n_tok_fp8 = sum(len(o.outputs[0].token_ids) for o in out_fp8)
    print(f"FP8 KV cache:  {time_fp8:.2f} s, {n_tok_fp8} tokens, {n_tok_fp8/time_fp8:.1f} tok/s")
    print(f"Speedup: {time_bf16/time_fp8:.2f}x")
except Exception as e:
    print(f"FP8 not available on this GPU: {e}")
    print("Use kv_cache_dtype='auto' (bf16) as baseline.")